# Module 11: Feature Drift Monitoring

## Learning Objectives

By completing this notebook, you will:
1. Implement PSI and KS-test drift detectors from first principles
2. Simulate realistic feature drift by shifting distributions over time
3. Build an automated re-selection trigger that fires when drift thresholds are crossed
4. Visualise drift scores over time with alert thresholds on a monitoring dashboard
5. Measure model performance before and after adaptive re-selection

## Prerequisites
- Guide 02 (PSI, KS test, Wasserstein concepts)
- Notebook 01 (fitted pipeline to monitor)
- scipy.stats basics

## Estimated Time: 15 minutes

---

## 1. Setup: Create a Drift Simulation Dataset

We simulate a commodity-like dataset where features drift over a 12-month production window. Three types of drift are introduced:

- **Mean shift**: Feature mean shifts linearly over time (like inflation affecting prices)
- **Variance expansion**: Feature volatility increases over time (like a market stress period)
- **Distribution shape change**: A normally distributed feature transitions to right-skewed (like default rates during a recession)

The remaining features remain stable as a control group.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.stats import ks_2samp, wasserstein_distance, norm as scipy_norm
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

# --- Simulation parameters ---
N_TRAIN      = 2000    # training distribution reference size
N_PER_MONTH  = 200     # new observations per month in production
N_MONTHS     = 12      # production window
N_FEATURES   = 10      # total features
N_DRIFTING   = 4       # features that will drift
N_STABLE     = N_FEATURES - N_DRIFTING

FEATURE_NAMES = (
    [f'stable_{i}' for i in range(N_STABLE)] +
    ['drift_mean_shift', 'drift_variance', 'drift_shape', 'drift_combined']
)

def generate_reference(n: int, seed: int = 0) -> pd.DataFrame:
    """Generate reference (training) distribution."""
    rng = np.random.default_rng(seed)
    data = {}
    for i in range(N_STABLE):
        data[f'stable_{i}'] = rng.normal(0, 1, n)
    # All drifting features start at the same reference distribution
    data['drift_mean_shift'] = rng.normal(0, 1, n)
    data['drift_variance']   = rng.normal(0, 1, n)
    data['drift_shape']      = rng.normal(0, 1, n)
    data['drift_combined']   = rng.normal(0, 1, n)
    return pd.DataFrame(data)


def generate_month(month: int, n: int = N_PER_MONTH, seed: int = 0) -> pd.DataFrame:
    """
    Generate one month of production data.

    Drift increases linearly with month index:
      - drift_mean_shift: mean shifts by +0.3 per month
      - drift_variance:   std increases by 0.2 per month
      - drift_shape:      transitions from normal to log-normal
      - drift_combined:   both mean shift and variance expansion
    """
    rng = np.random.default_rng(seed + month * 1000)
    data = {}
    for i in range(N_STABLE):
        data[f'stable_{i}'] = rng.normal(0, 1, n)   # no drift

    # Drift 1: linear mean shift
    mean_shift = 0.3 * month
    data['drift_mean_shift'] = rng.normal(mean_shift, 1, n)

    # Drift 2: variance expansion
    std_factor = 1.0 + 0.2 * month
    data['drift_variance'] = rng.normal(0, std_factor, n)

    # Drift 3: shape change (blend from normal to log-normal)
    blend = min(month / N_MONTHS, 1.0)
    normal_part   = rng.normal(0, 1, n)
    lognormal_part = np.log(np.abs(rng.normal(0, 1, n)) + 0.1)
    data['drift_shape'] = (1 - blend) * normal_part + blend * lognormal_part

    # Drift 4: combined
    data['drift_combined'] = rng.normal(mean_shift * 0.5, std_factor * 0.8, n)

    return pd.DataFrame(data)


# Generate reference and 12 months of production data
reference_df = generate_reference(N_TRAIN)
monthly_data = [generate_month(m, seed=42) for m in range(N_MONTHS)]

print(f"Reference data: {reference_df.shape}")
print(f"Monthly production data: {len(monthly_data)} months × {N_PER_MONTH} observations")
print(f"Features: {FEATURE_NAMES}")

## 2. PSI Drift Detector

PSI measures how much the distribution of each feature has shifted from the reference. The key implementation detail is that **bins must be defined on the reference distribution** and applied identically to the current distribution.

In [ ]:
def compute_psi(
    reference: np.ndarray,
    current: np.ndarray,
    n_bins: int = 10,
    eps: float = 1e-6,
) -> float:
    """
    Population Stability Index between reference and current distributions.

    PSI = sum((cur_i - ref_i) * log(cur_i / ref_i)) across B bins

    Thresholds:
        < 0.10: stable
        0.10 – 0.20: warning
        > 0.20: critical

    CRITICAL: bins are defined on reference distribution only.
    Defining bins separately on reference and current makes PSI meaningless.
    """
    # Percentile-based bins anchored to reference distribution
    bin_edges = np.percentile(reference, np.linspace(0, 100, n_bins + 1))
    bin_edges[0]  -= 1e-10  # include minimum value
    bin_edges[-1] += 1e-10  # include maximum value

    ref_counts = np.histogram(reference, bins=bin_edges)[0]
    cur_counts = np.histogram(current,   bins=bin_edges)[0]

    # Proportions with smoothing to avoid log(0)
    ref_props = (ref_counts / ref_counts.sum()) + eps
    cur_props = (cur_counts / cur_counts.sum()) + eps
    # Renormalise after smoothing
    ref_props /= ref_props.sum()
    cur_props /= cur_props.sum()

    return float(np.sum((cur_props - ref_props) * np.log(cur_props / ref_props)))


def compute_psi_all_features(
    reference: pd.DataFrame,
    current: pd.DataFrame,
    n_bins: int = 10,
) -> pd.DataFrame:
    """Compute PSI for every feature column."""
    rows = []
    for col in reference.columns:
        psi_val = compute_psi(reference[col].values, current[col].values, n_bins)
        if psi_val > 0.20:
            status = 'critical'
        elif psi_val > 0.10:
            status = 'warning'
        else:
            status = 'ok'
        rows.append({'feature': col, 'psi': psi_val, 'status': status})
    return pd.DataFrame(rows).sort_values('psi', ascending=False)


# Test PSI at month 0 and month 11
psi_month_0  = compute_psi_all_features(reference_df, monthly_data[0])
psi_month_11 = compute_psi_all_features(reference_df, monthly_data[11])

print("PSI at Month 0 (should be near-zero for all features):")
print(psi_month_0[['feature', 'psi', 'status']].to_string(index=False))
print()
print("PSI at Month 11 (drifting features should be critical):")
print(psi_month_11[['feature', 'psi', 'status']].to_string(index=False))

## 3. KS Test Drift Detector

The two-sample KS test provides formal p-values for distribution changes. When monitoring many features simultaneously, apply Bonferroni correction to control the family-wise error rate.

In [ ]:
def ks_drift_all_features(
    reference: pd.DataFrame,
    current: pd.DataFrame,
    alpha: float = 0.05,
) -> pd.DataFrame:
    """
    Two-sample KS test for all features with Bonferroni correction.

    Parameters
    ----------
    alpha : float
        Family-wise significance level (Bonferroni correction applied).
    """
    n_features = len(reference.columns)
    # Bonferroni correction: divide alpha by number of tests
    alpha_corrected = alpha / n_features

    rows = []
    for col in reference.columns:
        stat, pval = ks_2samp(reference[col].values, current[col].values)
        rows.append({
            'feature':    col,
            'ks_stat':    round(float(stat), 4),
            'pvalue':     round(float(pval), 4),
            'drifted':    pval < alpha_corrected,
        })

    df = pd.DataFrame(rows).sort_values('ks_stat', ascending=False)
    n_drifted = df['drifted'].sum()
    print(f"KS Test (Bonferroni alpha={alpha_corrected:.4f}): "
          f"{n_drifted}/{n_features} features show significant drift")
    return df


print("KS Drift Test at Month 6:")
ks_month_6 = ks_drift_all_features(reference_df, monthly_data[6])
print(ks_month_6.to_string(index=False))

## 4. Track Drift Over Time

Run both detectors for all 12 months and record the results. This produces the time series of drift metrics that feeds the monitoring dashboard and the re-selection trigger.

In [ ]:
# Compute PSI and KS for every month
monthly_psi_records = []
monthly_ks_records  = []

for month_idx, month_df in enumerate(monthly_data):
    psi_report = compute_psi_all_features(reference_df, month_df)
    ks_report  = ks_drift_all_features(reference_df, month_df, alpha=0.05)

    for _, row in psi_report.iterrows():
        monthly_psi_records.append({
            'month': month_idx,
            'feature': row['feature'],
            'psi': row['psi'],
            'status': row['status'],
        })
    for _, row in ks_report.iterrows():
        monthly_ks_records.append({
            'month': month_idx,
            'feature': row['feature'],
            'ks_stat': row['ks_stat'],
            'drifted': row['drifted'],
        })

psi_timeline = pd.DataFrame(monthly_psi_records)
ks_timeline  = pd.DataFrame(monthly_ks_records)

print(f"PSI timeline: {psi_timeline.shape[0]} records")
print(f"KS timeline:  {ks_timeline.shape[0]} records")

# Summarise: count of features above PSI thresholds per month
monthly_summary = psi_timeline.groupby('month').agg(
    n_critical=('status', lambda x: (x == 'critical').sum()),
    n_warning=('status', lambda x: (x == 'warning').sum()),
    n_ok=('status', lambda x: (x == 'ok').sum()),
    mean_psi=('psi', 'mean'),
    max_psi=('psi', 'max'),
).reset_index()

print("\nMonthly drift summary:")
print(monthly_summary.to_string(index=False))

## 5. Drift Monitoring Dashboard

Visualise the drift progression with three complementary charts:
1. **PSI heatmap**: Shows which features are drifting and when
2. **Critical feature count**: Shows when the re-selection trigger should fire
3. **Feature-level PSI trajectories**: Shows drift speed for drifting vs. stable features

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# --- Chart 1: PSI heatmap (features × months) ---
psi_pivot = psi_timeline.pivot(index='feature', columns='month', values='psi')
# Order: stable features first, then drifting
stable_features  = [f for f in psi_pivot.index if 'stable' in f]
drifting_features = [f for f in psi_pivot.index if 'drift' in f]
psi_pivot = psi_pivot.loc[stable_features + drifting_features]

im = axes[0].imshow(
    psi_pivot.values, aspect='auto', cmap='RdYlGn_r', vmin=0, vmax=0.5
)
plt.colorbar(im, ax=axes[0], label='PSI')
axes[0].set_xticks(range(N_MONTHS))
axes[0].set_xticklabels([f'M{i}' for i in range(N_MONTHS)])
axes[0].set_yticks(range(len(psi_pivot.index)))
axes[0].set_yticklabels(psi_pivot.index, fontsize=9)
axes[0].set_title('Feature PSI Over Time (green=stable, red=critical)', fontsize=12)
# Add divider between stable and drifting features
axes[0].axhline(y=len(stable_features) - 0.5, color='black', linewidth=2,
                 linestyle='--', label='stable/drifting boundary')

# Threshold lines on heatmap
for i, feat in enumerate(psi_pivot.index):
    for j in range(N_MONTHS):
        val = psi_pivot.iloc[i, j]
        axes[0].text(j, i, f'{val:.2f}', ha='center', va='center',
                      fontsize=6, color='black' if val < 0.3 else 'white')

# --- Chart 2: Count of critical features per month (trigger signal) ---
trigger_threshold_count = N_FEATURES * 0.2  # 20% of features critical = trigger
bar_colors = [
    '#d62728' if n >= trigger_threshold_count else '#1f77b4'
    for n in monthly_summary['n_critical']
]
axes[1].bar(monthly_summary['month'], monthly_summary['n_critical'],
             color=bar_colors, edgecolor='white', alpha=0.85)
axes[1].axhline(y=trigger_threshold_count, color='orange', linestyle='--',
                 linewidth=2, label=f'Re-selection trigger (20% = {trigger_threshold_count:.0f} features)')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('# Features PSI > 0.20')
axes[1].set_title('Re-Selection Trigger Signal (red bars = trigger fires)', fontsize=12)
axes[1].legend()
axes[1].set_xticks(range(N_MONTHS))
axes[1].set_xticklabels([f'M{i}' for i in range(N_MONTHS)])

# --- Chart 3: Per-feature PSI trajectories ---
for feat in drifting_features:
    feat_psi = psi_timeline[psi_timeline['feature'] == feat]
    axes[2].plot(feat_psi['month'], feat_psi['psi'],
                  marker='o', linewidth=2, label=feat, markersize=4)

# Show stable feature envelope
stable_psi = psi_timeline[psi_timeline['feature'].str.contains('stable')]
stable_max = stable_psi.groupby('month')['psi'].max()
axes[2].fill_between(stable_max.index, 0, stable_max.values,
                      alpha=0.15, color='green', label='stable features range')

axes[2].axhline(y=0.10, color='orange', linestyle=':', linewidth=1.5, label='warning (0.10)')
axes[2].axhline(y=0.20, color='red',    linestyle=':', linewidth=1.5, label='critical (0.20)')
axes[2].set_xlabel('Month')
axes[2].set_ylabel('PSI')
axes[2].set_title('PSI Trajectory: Drifting vs. Stable Features', fontsize=12)
axes[2].legend(loc='upper left', fontsize=8)
axes[2].set_xticks(range(N_MONTHS))
axes[2].set_xticklabels([f'M{i}' for i in range(N_MONTHS)])

plt.suptitle('Feature Drift Monitoring Dashboard', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 6. Automated Re-Selection Trigger

The trigger monitors PSI and KS metrics each month. When thresholds are crossed, it signals that re-selection should occur. We simulate what happens when re-selection fires at month 4 (when drift is first detected) vs. no re-selection.

In [ ]:
import datetime
from dataclasses import dataclass, field


@dataclass
class DriftTrigger:
    """
    Monitor PSI and KS metrics and fire re-selection when thresholds are crossed.
    """
    psi_threshold:          float = 0.20   # PSI > this is 'critical'
    psi_fraction_threshold: float = 0.20   # trigger if > 20% of features are critical
    ks_alpha:               float = 0.05   # KS significance level (Bonferroni applied)
    ks_fraction_threshold:  float = 0.30   # trigger if > 30% of features drifted
    max_months_since_resel: int   = 6      # scheduled: re-select every 6 months
    last_reselect_month:    int   = 0
    trigger_log:            list  = field(default_factory=list)

    def check(
        self,
        month: int,
        reference: pd.DataFrame,
        current: pd.DataFrame,
    ) -> tuple[bool, str]:
        """
        Evaluate all trigger conditions for the current month.

        Returns
        -------
        (should_reselect, reason)
        """
        n_features = len(reference.columns)

        # 1. Scheduled trigger (safety net)
        if month - self.last_reselect_month >= self.max_months_since_resel:
            reason = f"Scheduled trigger (month {month})"
            self.trigger_log.append({'month': month, 'reason': reason})
            return True, reason

        # 2. PSI trigger
        psi_report = compute_psi_all_features(reference, current)
        n_critical = (psi_report['psi'] > self.psi_threshold).sum()
        if n_critical / n_features >= self.psi_fraction_threshold:
            worst = psi_report.iloc[0]
            reason = (
                f"PSI trigger: {n_critical}/{n_features} features critical "
                f"(worst: {worst['feature']} PSI={worst['psi']:.3f})"
            )
            self.trigger_log.append({'month': month, 'reason': reason})
            return True, reason

        # 3. KS trigger
        ks_report = ks_drift_all_features(reference, current, self.ks_alpha)
        n_drifted = ks_report['drifted'].sum()
        if n_drifted / n_features >= self.ks_fraction_threshold:
            reason = (
                f"KS trigger: {n_drifted}/{n_features} features drifted "
                f"(alpha={self.ks_alpha})"
            )
            self.trigger_log.append({'month': month, 'reason': reason})
            return True, reason

        return False, "No trigger — all metrics within thresholds"

    def record_reselection(self, month: int):
        """Update the last re-selection timestamp."""
        self.last_reselect_month = month


# Run trigger checks across all 12 months
trigger = DriftTrigger(
    psi_threshold=0.20,
    psi_fraction_threshold=0.20,   # 20% of features critical
    ks_fraction_threshold=0.30,
    max_months_since_resel=6,
)

print("Month-by-month trigger evaluation:")
print("-" * 60)
for month in range(N_MONTHS):
    fire, reason = trigger.check(month, reference_df, monthly_data[month])
    if fire:
        print(f"Month {month:2d}: TRIGGER FIRES — {reason}")
        trigger.record_reselection(month)
    else:
        print(f"Month {month:2d}: OK       — {reason}")

## 7. Model Performance Before and After Re-Selection

We simulate model performance over time using a linear scoring model. The model degrades as drift accumulates. After re-selection fires at month 4, we retrain on recent data and show the performance recovery.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score


def generate_labels(X: pd.DataFrame, noise: float = 0.3, seed: int = 0) -> np.ndarray:
    """
    Generate binary labels from a simple linear combination of features.
    The true relationship uses stable features only (concept is stable;
    only feature distributions drift).
    """
    rng = np.random.default_rng(seed)
    # True signal: linear combination of stable features
    stable_cols = [c for c in X.columns if 'stable' in c]
    logit = X[stable_cols].values @ np.ones(len(stable_cols)) + rng.normal(0, noise, len(X))
    prob  = 1 / (1 + np.exp(-logit))
    return (prob > 0.5).astype(int)


# Generate labels for reference and all months
y_ref   = generate_labels(reference_df, seed=0)
y_monthly = [generate_labels(m, seed=i) for i, m in enumerate(monthly_data)]

# --- Model A: no re-selection, trained once on reference data ---
pipe_static = Pipeline([
    ('scaler', StandardScaler()),
    ('model',  GradientBoostingClassifier(n_estimators=50, random_state=42)),
])
pipe_static.fit(reference_df, y_ref)

# --- Model B: re-selects features when triggered (month 4 and 6) ---
# Simulate re-selection at months 4 and 6 using recent 3 months of data
RESELECT_MONTHS = [4, 6]
pipe_adaptive = None
adaptive_features = list(reference_df.columns)  # start with all features

# --- Score both models over 12 months ---
auc_static   = []
auc_adaptive = []

for month in range(N_MONTHS):
    month_df = monthly_data[month]
    month_y  = y_monthly[month]

    # Static model scores on current month
    y_prob_static = pipe_static.predict_proba(month_df)[:, 1]
    auc_static.append(roc_auc_score(month_y, y_prob_static))

    # Adaptive model: re-train on trigger months
    if month in RESELECT_MONTHS:
        # Collect last 3 months of data as re-training set
        recent_months = list(range(max(0, month - 2), month + 1))
        X_recent = pd.concat([monthly_data[m] for m in recent_months], ignore_index=True)
        y_recent = np.concatenate([y_monthly[m] for m in recent_months])

        # Re-select: use only stable-appearing features (simplified: all features)
        pipe_adaptive = Pipeline([
            ('scaler', StandardScaler()),
            ('model',  GradientBoostingClassifier(n_estimators=50, random_state=42)),
        ])
        pipe_adaptive.fit(X_recent, y_recent)
        print(f"Month {month}: Re-selection fired. Model retrained on "
              f"{len(X_recent)} recent observations.")

    if pipe_adaptive is not None:
        y_prob_adaptive = pipe_adaptive.predict_proba(month_df)[:, 1]
        auc_adaptive.append(roc_auc_score(month_y, y_prob_adaptive))
    else:
        # Before first re-selection, use static model
        auc_adaptive.append(auc_static[-1])

# --- Plot AUC comparison ---
fig, ax = plt.subplots(figsize=(12, 5))

months = list(range(N_MONTHS))
ax.plot(months, auc_static,   'b-o', linewidth=2, label='Static model (no re-selection)',
         markersize=6)
ax.plot(months, auc_adaptive, 'g-s', linewidth=2, label='Adaptive model (with re-selection)',
         markersize=6)

# Mark re-selection events
for rs_month in RESELECT_MONTHS:
    ax.axvline(x=rs_month, color='red', linestyle='--', alpha=0.7)
    ax.text(rs_month + 0.1, ax.get_ylim()[0] + 0.01,
             f'Re-select\nM{rs_month}', color='red', fontsize=9)

ax.set_xlabel('Production Month', fontsize=12)
ax.set_ylabel('ROC AUC', fontsize=12)
ax.set_title('Model AUC: Static vs. Adaptive Re-Selection', fontsize=13)
ax.legend(fontsize=11)
ax.set_xticks(months)
ax.set_xticklabels([f'M{m}' for m in months])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nFinal month AUC — Static: {auc_static[-1]:.4f} | Adaptive: {auc_adaptive[-1]:.4f}")
print(f"AUC improvement from re-selection: {auc_adaptive[-1] - auc_static[-1]:+.4f}")

## 8. Exercise: Wasserstein Distance Monitor

**Task:** Implement a `wasserstein_drift_report` function that:
1. Computes the 1-Wasserstein distance for each feature
2. Normalises by the reference standard deviation (so distances are comparable across features)
3. Returns a DataFrame sorted by normalised Wasserstein descending
4. Adds a column `severity` that classifies each feature as 'low' (<0.5), 'medium' (0.5-1.0), or 'high' (>1.0) normalised distance

Then use it to rank the drifting features at month 11 by drift severity.

**Hint:** `scipy.stats.wasserstein_distance(u_values, v_values)` computes the 1-Wasserstein distance between two 1D distributions.

In [ ]:
def wasserstein_drift_report(reference: pd.DataFrame, current: pd.DataFrame) -> pd.DataFrame:
    """
    Compute normalised Wasserstein distance for each feature.

    Returns a DataFrame with columns:
      - feature
      - wasserstein_raw     : raw 1-Wasserstein distance
      - wasserstein_norm    : normalised by reference std (comparable across features)
      - severity            : 'low' / 'medium' / 'high'

    Sorted by wasserstein_norm descending.
    """
    # TODO: Implement
    # For each column:
    #   1. Compute wasserstein_distance(reference[col], current[col])
    #   2. Normalise by reference[col].std() (handle std=0 case)
    #   3. Classify severity: low (<0.5), medium (0.5-1.0), high (>1.0)
    raise NotImplementedError("Implement wasserstein_drift_report")


# Self-check tests
def test_wasserstein_report():
    report = wasserstein_drift_report(reference_df, monthly_data[11])

    assert isinstance(report, pd.DataFrame), "Return type must be pd.DataFrame"
    assert list(report.columns) == ['feature', 'wasserstein_raw', 'wasserstein_norm', 'severity'], (
        f"Expected columns ['feature', 'wasserstein_raw', 'wasserstein_norm', 'severity'], "
        f"got {list(report.columns)}"
    )
    assert len(report) == len(reference_df.columns), "One row per feature required"

    # Drifting features should rank higher than stable ones
    top_features = report.head(4)['feature'].tolist()
    n_drifting_in_top = sum(1 for f in top_features if 'drift' in f)
    assert n_drifting_in_top >= 3, (
        f"Expected at least 3 drifting features in top-4, got {n_drifting_in_top}: {top_features}"
    )

    # Severity classification
    assert set(report['severity'].unique()).issubset({'low', 'medium', 'high'}), (
        "Severity must be 'low', 'medium', or 'high'"
    )

    print("All tests passed!")
    print(report.to_string(index=False))

# Uncomment to run after implementing:
# test_wasserstein_report()

## 9. Summary

### Key Takeaways

1. **PSI** is the industry standard for operational drift monitoring. Bins must always be defined on the reference distribution. Thresholds 0.10 (warning) and 0.20 (critical) are established conventions from credit scoring.

2. **KS test** provides formal p-values. Apply Bonferroni correction when testing multiple features simultaneously to control the false positive rate.

3. **Wasserstein distance** quantifies drift magnitude continuously, enabling severity ranking of drifted features. Normalise by reference standard deviation for cross-feature comparability.

4. **Re-selection triggers** should combine multiple signals: PSI/KS for early warning, scheduled triggers as safety nets, and performance monitoring for confirmation.

5. **Adaptive re-selection** measurably improves long-run model performance compared to static models that never adapt to distribution shift.

### What's Next

Notebook 03 covers **MLflow tracking**: logging every re-selection run so you can compare, reproduce, and audit all selection decisions.

### Additional Resources
- Siddiqi (2006): *Credit Risk Scorecards* — PSI origin and credit scoring context
- Gama et al. (2014): A Survey on Concept Drift Adaptation — *ACM Computing Surveys*
- `evidently` library: open-source drift monitoring — https://evidentlyai.com/